# 🪆 Python Inner Functions — The Practical Guide
### *What they are, why they exist, and when to use them*

---

> **Mental Model First:**
> An inner function is a **private helper** that lives inside another function.
> It can see everything in its parent's room — variables, parameters, all of it.
> The outside world can't see it or call it directly.
> It exists only to serve its parent.

---

## 📋 Table of Contents

| # | Section |
|---|---|
| 1 | [The Basics — Syntax and Scope](#1) |
| 2 | [The Closure Rule — What Can You Touch?](#2) |
| 3 | [Use Case 1: Private Helper (Backtracking)](#3) |
| 4 | [Use Case 2: Avoid Repeating Code (DRY)](#4) |
| 5 | [Use Case 3: Closure — Capture State](#5) |
| 6 | [Use Case 4: Decorator Pattern](#6) |
| 7 | [The nonlocal Keyword](#7) |
| 8 | [Cheat Sheet](#8) |

<a id='1'></a>
## 1. The Basics — Syntax and Scope

---

```
  def outer(x):          ← outer function
      y = 10             ← outer variable

      def inner():       ← inner function — defined INSIDE outer
          print(x, y)   ← can see both x and y — no problem

      inner()            ← outer calls inner
      # inner() only exists inside outer — invisible to the world

  SCOPE LAYERS (innermost wins):
  Local → Enclosing → Global → Built-in   (LEGB rule)

  inner can READ from every layer above it.
  inner can MUTATE objects (list.append, obj.attr = x) from any layer.
  inner CANNOT reassign a variable from an outer layer without nonlocal.
```

In [ ]:
def outer(x):
    y = 10                    # outer variable — inner can see this

    def inner():
        print(f"x={x} y={y}")  # reading outer variables — perfectly fine

    inner()                   # outer calls inner
    # inner does not exist here from the outside — try calling it and you get NameError

outer(5)    # prints: x=5 y=10
print()

# inner() here would raise NameError — it only lives inside outer
try:
    inner()
except NameError as e:
    print(f"Calling inner() outside: {e}")

<a id='2'></a>
## 2. The Closure Rule — What Can You Touch?

---

```
  OPERATION            ALLOWED?   WHY
  ──────────────────────────────────────────────────────
  result.append(x)     ✅ YES     mutating the object — reference unchanged
  result[0] = 5        ✅ YES     mutating via bracket — reference unchanged
  obj.value = 10       ✅ YES     setting an attribute — reference unchanged
  count + 1            ✅ YES     reading — always fine

  count = count + 1    ❌ NO      reassignment — shadows outer, creates local
  result = []          ❌ NO      reassignment — new local, outer unchanged
  flag = True          ❌ NO      reassignment — needs nonlocal
  ──────────────────────────────────────────────────────

  THE RULE IN ONE LINE:
  dot or bracket = mutation = fine
  equals sign on a plain variable = reassignment = needs nonlocal
```

In [ ]:
def demonstrate_closure_rules():
    result = []        # a list — inner can mutate it
    count  = 0         # an int — inner CANNOT reassign it without nonlocal

    def inner():
        result.append(42)   # ✅ mutation — fine, result is still the same list object
        # count = count + 1  # ❌ would raise UnboundLocalError — Python sees assignment,
                             # treats count as local, then tries to read it before assign

    inner()
    print("result after inner():", result)   # [42]
    print("count after inner(): ", count)    # 0 — unchanged

demonstrate_closure_rules()
print()

# Proof that mutation works vs reassignment fails
def mutation_vs_reassign():
    my_list = [1, 2, 3]

    def mutate():
        my_list.append(99)      # fine — same object, just adding to it
        my_list[0] = 100        # fine — same object, just changing a slot

    def reassign():
        my_list = [999]         # creates a NEW local variable named my_list
                                # outer my_list is untouched

    mutate()
    print("after mutate():", my_list)    # [100, 2, 3, 99]

    reassign()
    print("after reassign():", my_list)  # [100, 2, 3, 99] — unchanged!

mutation_vs_reassign()

<a id='3'></a>
## 3. Use Case 1: Private Helper — Backtracking (LC 39)

---

The most common interview use of inner functions.
The recursive helper needs access to `candidates`, `result`, and `target`.
Instead of passing them as parameters every call, the inner function just sees them.

```
  WITHOUT inner function — ugly, all params passed explicitly:
  def backtrack(candidates, result, target, start, current, remaining):
      ...
  backtrack(candidates, result, target, 0, [], target)

  WITH inner function — clean, only the changing state is passed:
  def backtrack(start, current, remaining):
      ...
  backtrack(0, [], target)
```

In [ ]:
from typing import List

def combinationSum(candidates: List[int], target: int) -> List[List[int]]:
    """
    LC 39 — Combination Sum
    Approach: backtracking. Inner function sees candidates, target, result
    via closure — only the changing state (start, current, remaining) is passed.
    Time:  O(n^(t/m)) — n=candidates, t=target, m=min candidate
    Space: O(t/m)     — max recursion depth
    """
    result = []    # inner will append to this — mutation, no nonlocal needed

    def backtrack(start, current, remaining):
        if remaining == 0:
            result.append(current[:])   # found valid combo — snapshot current and save
            return
        if remaining < 0:
            return                      # overshot — prune this branch

        for i in range(start, len(candidates)):
            current.append(candidates[i])              # choose
            backtrack(i, current, remaining - candidates[i])  # explore (i not i+1 — reuse allowed)
            current.pop()                              # un-choose — backtrack

    backtrack(0, [], target)   # start = 0, current = empty, remaining = full target
    return result


def test_harness_cs(fn):
    tests = [
        ([2,3,6,7], 7,  sorted([sorted(x) for x in [[2,2,3],[7]]])),
        ([2,3,5],   8,  sorted([sorted(x) for x in [[2,2,2,2],[2,3,3],[3,5]]])),
        ([2],       1,  []),
        ([1],       1,  [[1]]),
        ([1],       2,  [[1,1]]),
    ]
    passed = 0
    for candidates, target, expected in tests:
        got = sorted([sorted(x) for x in fn(candidates, target)])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | candidates={candidates} target={target} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


print(combinationSum([2,3,6,7], 7))   # [[2,2,3],[7]]
print(combinationSum([2,3,5],   8))   # [[2,2,2,2],[2,3,3],[3,5]]
test_harness_cs(combinationSum)
print("combinationSum defined.")

<a id='4'></a>
## 4. Use Case 2: Avoid Repeating Code (DRY)

---

You have a chunk of logic that needs to run 2–3 times inside a function.
Instead of copy-pasting or writing a module-level helper, define it inline.
The inner function sees the outer variables — no extra parameters needed.

Classic example: LC 5 — Longest Palindromic Substring.
The expand logic runs twice per character (odd center, even center).
Defining it as an inner function avoids repetition and keeps it clean.

In [ ]:
def longestPalindrome(s: str) -> str:
    """
    LC 5 — Longest Palindromic Substring
    Inner function expand() runs the same logic for odd and even centers.
    Without it, the expand logic would be copy-pasted twice.
    Time:  O(n^2)  Space: O(1)
    """
    res = ""

    def expand(l, r):
        # sees `s` and `res` from outer — only l and r change per call
        result = ""
        while l >= 0 and r < len(s) and s[l] == s[r]:
            if (r - l + 1) > len(result):
                result = s[l:r+1]    # wider palindrome found — update
            l -= 1
            r += 1
        return result

    for i in range(len(s)):
        odd  = expand(i, i)       # odd length — single char center
        even = expand(i, i + 1)   # even length — gap between i and i+1
        candidate = odd if len(odd) >= len(even) else even
        if len(candidate) > len(res):
            res = candidate

    return res


print(longestPalindrome("babad"))    # bab or aba
print(longestPalindrome("cbbd"))     # bb
print(longestPalindrome("racecar"))  # racecar
print(longestPalindrome("a"))        # a
print("longestPalindrome defined.")

<a id='5'></a>
## 5. Use Case 3: Closure — Capture State

---

A closure is an inner function that **remembers** the outer variables
even after the outer function has finished running.
The inner function carries those variables with it like a backpack.

```
  outer() runs and returns inner.
  outer's local variables are gone... or are they?
  inner still holds a reference to them — that's the closure.
  Calling inner() later still works because the backpack is still there.
```

Practical use: factory functions, memoization, counters.

In [ ]:
# Example 1 — multiplier factory
# outer captures the multiplier, inner remembers it
def make_multiplier(factor):
    def multiply(x):
        return x * factor   # factor is captured from outer — lives in the closure
    return multiply         # return the function itself, not the result

double = make_multiplier(2)   # factor=2 is baked in
triple = make_multiplier(3)   # factor=3 is baked in

print(double(5))   # 10
print(triple(5))   # 15
print(double(9))   # 18
print()

# Example 2 — simple memoization using a closure dict
def make_memoized_fib():
    cache = {}              # cache lives in the closure — persists across calls

    def fib(n):
        if n in cache:
            return cache[n]     # already computed — cache hit
        if n <= 1:
            return n
        cache[n] = fib(n-1) + fib(n-2)   # cache.update = mutation = fine
        return cache[n]

    return fib

fib = make_memoized_fib()
print([fib(i) for i in range(10)])   # [0,1,1,2,3,5,8,13,21,34]

<a id='6'></a>
## 6. Use Case 4: Decorator Pattern

---

A decorator wraps a function to add behavior before or after it runs.
The inner function is the wrapper — it sees the original function via closure.

```
  @timer
  def my_func(): ...

  is exactly the same as:

  my_func = timer(my_func)
```

You'll see this in production code constantly — logging, timing, caching, auth checks.

In [ ]:
import time

def timer(func):
    # func is captured in the closure
    def wrapper(*args, **kwargs):
        start  = time.perf_counter()
        result = func(*args, **kwargs)    # call the original function
        end    = time.perf_counter()
        print(f"{func.__name__} took {(end-start)*1000:.3f} ms")
        return result
    return wrapper


@timer
def slow_sum(n):
    return sum(range(n))


@timer
def fast_sum(n):
    return n * (n - 1) // 2    # Gauss formula — O(1)


print(slow_sum(1_000_000))   # O(n) — measurably slower
print(fast_sum(1_000_000))   # O(1) — near zero

<a id='7'></a>
## 7. The nonlocal Keyword

---

```
  When you need to REASSIGN a scalar variable from the outer scope,
  you need to declare it nonlocal first.

  Without nonlocal:
  count = 0
  def inner():
      count = count + 1   ← UnboundLocalError
      # Python sees the assignment, treats count as local,
      # then tries to read it before it's assigned

  With nonlocal:
  count = 0
  def inner():
      nonlocal count
      count = count + 1   ← works — modifies outer count directly

  WHEN DO YOU NEED IT?
  - Incrementing a counter across recursive calls
  - Flipping a boolean flag
  - Updating a best/max scalar value

  WHEN DO YOU NOT NEED IT?
  - Appending to a list  (mutation, not reassignment)
  - Setting dict keys    (mutation)
  - Setting object attrs (mutation)
```

In [ ]:
# Without nonlocal — the workaround: use a list as a mutable container
def count_nodes_no_nonlocal(n):
    count = [0]           # list wraps the int — mutation allowed
    def dfs(node):
        if node <= 0:
            return
        count[0] += 1     # mutating the list slot — no nonlocal needed
        dfs(node - 1)
    dfs(n)
    return count[0]

print("count without nonlocal:", count_nodes_no_nonlocal(5))  # 5
print()

# With nonlocal — cleaner
def count_nodes_nonlocal(n):
    count = 0
    def dfs(node):
        nonlocal count         # tell Python: count lives in the enclosing scope
        if node <= 0:
            return
        count += 1             # reassignment — works because of nonlocal
        dfs(node - 1)
    dfs(n)
    return count

print("count with nonlocal:   ", count_nodes_nonlocal(5))   # 5
print()

# Real interview example — diameter of binary tree needs nonlocal
# The best diameter seen so far is a scalar updated across all recursive calls
def fake_diameter_demo():
    best = 0

    def dfs(depth):
        nonlocal best
        if depth <= 0:
            return 0
        left  = dfs(depth - 1)
        right = dfs(depth - 2) if depth > 1 else 0
        best = max(best, left + right)   # reassigning best — needs nonlocal
        return max(left, right) + 1

    dfs(4)
    return best

print("best diameter demo:", fake_diameter_demo())

<a id='8'></a>
## 8. 📋 Cheat Sheet

---

### When to use an inner function:

| Situation | Pattern |
|---|---|
| Recursive helper needs outer state | Inner function + closure (no extra params) |
| Same logic runs 2+ times inside one function | Inner function (DRY) |
| Need a function that remembers a value | Closure (factory pattern) |
| Need to wrap a function with extra behavior | Inner function as decorator |
| Need to count/track across recursive calls | Inner + nonlocal (or list wrapper) |

---

### The mutation vs reassignment rule:

```python
result.append(x)     # ✅ mutation  — no nonlocal
result[i] = x        # ✅ mutation  — no nonlocal
obj.attr  = x        # ✅ mutation  — no nonlocal
cache[key] = val     # ✅ mutation  — no nonlocal

count = count + 1    # ❌ reassign  — needs nonlocal count
best  = max(best, x) # ❌ reassign  — needs nonlocal best
flag  = True         # ❌ reassign  — needs nonlocal flag
```

---

### The list-wrapper workaround (avoid nonlocal):

```python
# Instead of:
count = 0
def inner():
    nonlocal count
    count += 1

# You can do:
count = [0]          # wrap in a list
def inner():
    count[0] += 1    # mutation — no nonlocal needed
```

Both work. `nonlocal` is cleaner to read. The list wrapper avoids the keyword.
Pick one and be consistent.

---

### Gotchas:

```
❌  Calling inner() from outside outer() → NameError — it doesn't exist there
❌  Reassigning an outer variable without nonlocal → UnboundLocalError
❌  Forgetting current[:] in backtracking → appending a reference, not a snapshot
✅  result.append(current[:])  ← always snapshot with [:] when saving combos
✅  Inner functions can call other inner functions defined in the same outer scope
✅  nonlocal only goes one level up — use it at the level that owns the variable
```

---
*End of Inner Functions Guide — Sean Edition*